# 02 — Crash-to-LION Matching and Safety Metrics

Processes the crash-to-roadway nearest-neighbor join and creates one safety-metric record per LION `SegmentID`.

The nearest-neighbor join used in the original workflow was created in QGIS from valid crash points and the EPSG:2263 LION street-centerline layer. Tied nearest matches can create duplicate crash rows, so this notebook keeps one deterministic nearest match per `COLLISION_ID`.

Matches farther than **50 ft** are excluded from segment-level safety metrics.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent

raw = project / "data" / "raw"
interim = project / "data" / "interim"
processed = project / "data" / "processed"
tables_dir = project / "outputs" / "tables"

interim.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project)

In [ ]:
crash_join_file = interim / "crashes_with_segment_ids.csv"

# backward-compatible location used during development.
if not crash_join_file.exists():
    crash_join_file = project / "docs" / "crashes_with_segment_ids.csv"

crash_join = pd.read_csv(crash_join_file, low_memory=False)

print("Input rows:", len(crash_join))
print("Unique crashes:", crash_join["COLLISION_ID"].nunique())

In [ ]:
crash_join["SegmentID"] = pd.to_numeric(
    crash_join["SegmentID"],
    errors="coerce"
).astype("Int64")

crash_join["distance"] = pd.to_numeric(
    crash_join["distance"],
    errors="coerce"
)

crash_join_sorted = crash_join.sort_values(
    ["COLLISION_ID", "distance", "SegmentID"],
    na_position="last"
)

crash_join_dedup = (
    crash_join_sorted
    .drop_duplicates(subset="COLLISION_ID", keep="first")
    .copy()
)

print("Rows after tie deduplication:", len(crash_join_dedup))
print("Duplicate crashes:", crash_join_dedup["COLLISION_ID"].duplicated().sum())
print("Missing SegmentID:", crash_join_dedup["SegmentID"].isna().sum())
print(crash_join_dedup["distance"].describe())

In [ ]:
crashes_segment_final = crash_join_dedup[
    crash_join_dedup["SegmentID"].notna() &
    (crash_join_dedup["distance"] <= 50)
].copy()

print("Final segment-matched crashes:", len(crashes_segment_final))
print("Unique crashes:", crashes_segment_final["COLLISION_ID"].nunique())
print("Unique SegmentIDs:", crashes_segment_final["SegmentID"].nunique())
print("Maximum match distance:", crashes_segment_final["distance"].max())

In [ ]:
matched_out = processed / "crashes_segment_matched_2022_2026.csv"
crashes_segment_final.to_csv(matched_out, index=False)
print("Saved:", matched_out)

## Aggregate crash metrics by roadway segment

In [ ]:
crash_segment_metrics = (
    crashes_segment_final
    .groupby("SegmentID")
    .agg(
        crash_count=("COLLISION_ID", "count"),
        persons_injured=("NUMBER OF PERSONS INJURED", "sum"),
        persons_killed=("NUMBER OF PERSONS KILLED", "sum"),
        ped_injured=("NUMBER OF PEDESTRIANS INJURED", "sum"),
        ped_killed=("NUMBER OF PEDESTRIANS KILLED", "sum"),
        cyclist_injured=("NUMBER OF CYCLIST INJURED", "sum"),
        cyclist_killed=("NUMBER OF CYCLIST KILLED", "sum"),
        motorist_injured=("NUMBER OF MOTORIST INJURED", "sum"),
        motorist_killed=("NUMBER OF MOTORIST KILLED", "sum")
    )
    .reset_index()
)

crash_time_counts = (
    crashes_segment_final
    .pivot_table(
        index="SegmentID",
        columns="time_group",
        values="COLLISION_ID",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

crash_segment_metrics = crash_segment_metrics.merge(
    crash_time_counts,
    on="SegmentID",
    how="left"
)

crash_segment_metrics["vru_injured"] = (
    crash_segment_metrics["ped_injured"] +
    crash_segment_metrics["cyclist_injured"]
)

crash_segment_metrics["vru_killed"] = (
    crash_segment_metrics["ped_killed"] +
    crash_segment_metrics["cyclist_killed"]
)

print("Segment metric rows:", len(crash_segment_metrics))
print(crash_segment_metrics.head())

In [ ]:
metrics_out = processed / "crash_segment_metrics_2022_2026.csv"
crash_segment_metrics.to_csv(metrics_out, index=False)
print("Saved:", metrics_out)